# IMDB Movie Review Sentiment Classification

## Problem Statement

The purpose of this project is to classify IMDB movie reviews as **positive** or **negative**. The text is cleaned by converting it to lowercase, removing HTML, punctuation and stopwords, tokenizing the words, and applying stemming. TF-IDF is then used to convert the cleaned reviews into numerical features, and Logistic Regression is trained to predict sentiment.

**Dataset:** IMDB Dataset of 50K Movie Reviews  
**Target column:** `sentiment`


## 1. Import the Required Libraries

The libraries below are used for data handling, text cleaning, feature extraction, model training, and evaluation.


In [1]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

print("Libraries imported successfully.")


Libraries imported successfully.


## 2. Load the Dataset

Keep the downloaded CSV file in the same folder as this notebook. The Kaggle file is normally named `IMDB Dataset.csv`. The code also checks a few common filename variations.


In [2]:
possible_files = [
    "IMDB Dataset.csv",
    "IMDB_Dataset.csv",
    "imdb_dataset.csv",
    "IMDB_Dataset_CLEANED.csv"
]

data_file = None

for file_name in possible_files:
    if Path(file_name).exists():
        data_file = file_name
        break

if data_file is None:
    csv_files = list(Path(".").glob("*.csv"))
    if len(csv_files) == 1:
        data_file = str(csv_files[0])
    else:
        raise FileNotFoundError(
            "The IMDB CSV file was not found. Place 'IMDB Dataset.csv' "
            "in the same folder as this notebook."
        )

df = pd.read_csv(data_file)

print("Loaded file:", data_file)
print("Dataset shape:", df.shape)
display(df.head())


Loaded file: IMDB Dataset.csv
Dataset shape: (50000, 2)


,review,sentiment
0,"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as ...",positive
1,A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and g...,positive
2,"I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater...",positive
3,Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fightin...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is a visually stunning film to watch. Mr. Mattei offers us a vivid portr...",positive


## 3. Check the Dataset

This section checks the column names, missing values, duplicate rows, and the number of reviews in each sentiment class.


In [3]:
print("Column names:", df.columns.tolist())

print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing Values"))

print("Duplicate rows:", df.duplicated().sum())

print("\nSentiment distribution:")
display(df["sentiment"].value_counts().to_frame("Number of Reviews"))


Column names: ['review', 'sentiment']

Missing values:


,Missing Values
review,0
sentiment,0


Duplicate rows: 418

Sentiment distribution:


,Number of Reviews
sentiment,
positive,25000
negative,25000


## 4. Prepare the Data

Missing rows are removed because both the review and sentiment are required. Duplicate reviews are also removed so that repeated text does not influence the model more than once.


In [4]:
df = df[["review", "sentiment"]].copy()

df = df.dropna(subset=["review", "sentiment"])
df = df.drop_duplicates(subset=["review"]).reset_index(drop=True)

df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()
df = df[df["sentiment"].isin(["positive", "negative"])].reset_index(drop=True)

print("Shape after cleaning rows:", df.shape)
display(df["sentiment"].value_counts().to_frame("Number of Reviews"))


Shape after cleaning rows: (49582, 2)


,Number of Reviews
sentiment,
positive,24884
negative,24698


## 5. Full Text-Preprocessing Pipeline

For every review, the following steps are applied:

1. Convert the text to lowercase
2. Remove HTML tags
3. Remove punctuation, numbers, and special characters
4. Tokenize the review into individual words
5. Remove English stopwords
6. Apply Porter stemming
7. Join the cleaned words back into one string


In [5]:
stemmer = PorterStemmer()
stop_words = set(ENGLISH_STOP_WORDS)

# Negation words are useful for sentiment, so they are kept.
for word in ["not", "no", "nor"]:
    stop_words.discard(word)

def preprocess_text(text):
    # Convert the text to lowercase
    text = str(text).lower()

    # Remove HTML tags such as <br />
    text = re.sub(r"<.*?>", " ", text)

    # Remove website links
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove punctuation, numbers, and special characters
    text = re.sub(r"[^a-z']", " ", text)

    # Tokenize the review
    tokens = re.findall(r"[a-z]+", text)

    # Remove stopwords and very short tokens
    tokens = [
        word for word in tokens
        if word not in stop_words and len(word) > 1
    ]

    # Apply stemming
    stemmed_tokens = [stemmer.stem(word) for word in tokens]

    return " ".join(stemmed_tokens)

df["cleaned_review"] = df["review"].apply(preprocess_text)

# Remove rows that became empty after preprocessing
df = df[df["cleaned_review"].str.strip() != ""].reset_index(drop=True)

print("Text preprocessing completed.")
display(df[["review", "cleaned_review", "sentiment"]].head(3))


Text preprocessing completed.


,review,cleaned_review,sentiment
0,"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as ...",review mention watch just oz episod ll hook right exactli happen thing struck oz brutal unflinch scene violenc set r...,positive
1,A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and g...,wonder littl product film techniqu unassum old time bbc fashion give comfort discomfort sens realism entir piec acto...,positive
2,"I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater...",thought wonder way spend time hot summer weekend sit air condit theater watch light heart comedi plot simplist dialo...,positive


## 6. Encode the Sentiment Labels

The text labels are converted into numbers:

- Positive = 1
- Negative = 0


In [6]:
df["label"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})

display(df[["sentiment", "label"]].drop_duplicates().sort_values("label"))


,sentiment,label
3,negative,0
0,positive,1


## 7. Split the Data into Training and Testing Sets

The dataset is divided into 80% training data and 20% testing data. Stratification keeps the positive and negative class proportions consistent in both sets.


In [7]:
X = df["cleaned_review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

print("\nTraining label distribution:")
display(
    y_train.value_counts(normalize=True)
    .sort_index()
    .rename("Proportion")
    .to_frame()
)


Training reviews: 39665
Testing reviews: 9917

Training label distribution:


,Proportion
label,
0,0.498122
1,0.501878


## 8. Create the TF-IDF Feature Matrix

TF-IDF gives higher importance to useful words and lower importance to words that appear in many reviews. The vectorizer is fitted only on the training data to avoid information leakage.


In [8]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF matrix shape:", X_train_tfidf.shape)
print("Testing TF-IDF matrix shape:", X_test_tfidf.shape)
print("Number of TF-IDF features:", len(tfidf.get_feature_names_out()))


Training TF-IDF matrix shape: (39665, 20000)
Testing TF-IDF matrix shape: (9917, 20000)
Number of TF-IDF features: 20000


## 9. Train the Logistic Regression Classifier

Logistic Regression is suitable for high-dimensional text data and also provides coefficients that can be used to identify the most predictive words for each class.


In [9]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    solver="liblinear"
)

model.fit(X_train_tfidf, y_train)

print("Logistic Regression training completed.")


Logistic Regression training completed.


## 10. Evaluate the Model

The required evaluation measures are Accuracy and F1-score.


In [10]:
y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

results_table = pd.DataFrame({
    "Model": ["Logistic Regression"],
    "Accuracy": [accuracy],
    "F1-score": [f1]
})

display(
    results_table.style.format({
        "Accuracy": "{:.4f}",
        "F1-score": "{:.4f}"
    })
)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-score: {f1:.4f}")


,Model,Accuracy,F1-score
0,Logistic Regression,0.8980,0.8996


Accuracy: 0.8980
F1-score: 0.8996


### Classification Report

The classification report is included as additional information. It shows precision, recall, and F1-score for both sentiment classes.


In [11]:
report = classification_report(
    y_test,
    y_pred,
    target_names=["Negative", "Positive"],
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()
display(report_df.round(4))


,precision,recall,f1-score,support
Negative,0.9080,0.8848,0.8962,4940.000
Positive,0.8885,0.9110,0.8996,4977.000
accuracy,0.8980,0.8980,0.8980,0.898
macro avg,0.8982,0.8979,0.8979,9917.000
weighted avg,0.8982,0.8980,0.8979,9917.000


## 11. Top 10 Most Predictive Words per Class

For Logistic Regression:

- Large positive coefficients indicate words that strongly predict a positive review.
- Large negative coefficients indicate words that strongly predict a negative review.

The words below are taken directly from the trained classifier.


In [12]:
feature_names = np.array(tfidf.get_feature_names_out())
coefficients = model.coef_[0]

top_positive_indices = np.argsort(coefficients)[-10:][::-1]
top_negative_indices = np.argsort(coefficients)[:10]

top_words_table = pd.DataFrame({
    "Rank": range(1, 11),
    "Positive Predictive Word": feature_names[top_positive_indices],
    "Positive Coefficient": coefficients[top_positive_indices],
    "Negative Predictive Word": feature_names[top_negative_indices],
    "Negative Coefficient": coefficients[top_negative_indices]
})

display(
    top_words_table.style.format({
        "Positive Coefficient": "{:.4f}",
        "Negative Coefficient": "{:.4f}"
    })
)


,Rank,Positive Predictive Word,Positive Coefficient,Negative Predictive Word,Negative Coefficient
0,1,great,7.9512,worst,-9.5018
1,2,excel,7.2572,bad,-8.1186
2,3,best,5.7380,aw,-8.0050
3,4,enjoy,5.7079,wast,-7.5653
4,5,perfect,5.6494,bore,-6.9628
5,6,love,5.2992,poor,-6.1352
6,7,amaz,5.0250,terribl,-5.8561
7,8,favorit,4.7647,disappoint,-5.7149
8,9,fun,4.5398,fail,-5.2927
9,10,brilliant,4.3154,wors,-4.9853


## 12. Final Conclusion

The following conclusion is generated from the actual model results after the notebook is run.


In [13]:
print(
    f"The Logistic Regression model achieved an accuracy of {accuracy:.4f} "
    f"and an F1-score of {f1:.4f} on the test data. "
    "This shows that TF-IDF features can effectively represent the sentiment "
    "expressed in movie reviews. The predictive-word table also shows which "
    "words and phrases had the strongest influence on positive and negative predictions."
)


The Logistic Regression model achieved an accuracy of 0.8980 and an F1-score of 0.8996 on the test data. This shows that TF-IDF features can effectively represent the sentiment expressed in movie reviews. The predictive-word table also shows which words and phrases had the strongest influence on positive and negative predictions.


## Deliverables Checklist

- Full text-preprocessing pipeline: **Completed**
- Lowercase conversion: **Completed**
- Punctuation and special-character removal: **Completed**
- Stopword removal: **Completed**
- Tokenization: **Completed**
- Stemming: **Completed**
- TF-IDF feature matrix: **Completed**
- Trained classifier: **Completed**
- Accuracy: **Completed**
- F1-score: **Completed**
- Top 10 predictive words for each class: **Completed**
